In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import kpss
from statsmodels.tools.eval_measures import rmse
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose


In [ ]:
data = pd.read_csv('../dataset/FEDFUNDS.csv', 
                       parse_dates=['observation_date'], index_col='observation_date')

print("Samples:", len(data))

print(data.head())


# Graph
result = seasonal_decompose(data['FEDFUNDS'],  
                            model ='multiplicative')

result.plot()

# data["detrended"] = sm.OLS(data["FEDFUNDS"], sm.add_constant(range(len(data)))).fit().resid
# data["differenced"] = data["FEDFUNDS"].diff()

In [ ]:
# ADF Test
result_test = adfuller(data)

print(f"ADF Statistic: {result_test[0]}")
print(f"P-value: {result_test[1]}")
print(f"Critical Values: {result_test[4]}")

if result_test[1] < 0.05:
    print("Reject H0: Data is stationary")
else:
    print("Fail to reject H0: Data is non-stationary")

In [ ]:
# KPSS Test
def kpss_test(series, regression='c'):
    result_test = kpss(series, regression=regression)
    p_value = result_test[1]
    print(f"KPSS Statistic: {result_test[0]}")
    print(f"P-value: {result_test[1]}")
    print(f"Critical Values: {result_test[3]}")
    
    if p_value < 0.05:
        print("Reject H0: Data is non-stationary")
    else:
        print("Fail to Reject H0: Data is stationary")

kpss_test(data['FEDFUNDS'])

In [ ]:
# Graph
result = seasonal_decompose(data['FEDFUNDS'],  
                            model ='multiplicative')

result.plot()

# data["detrended"] = sm.OLS(data["FEDFUNDS"], sm.add_constant(range(len(data)))).fit().resid
# data["differenced"] = data["FEDFUNDS"].diff()


# ADF Test
result_test = adfuller(data)

print(f"ADF Statistic: {result_test[0]}")
print(f"P-value: {result_test[1]}")
print(f"Critical Values: {result_test[4]}")

if result_test[1] < 0.05:
    print("Reject H0: Data is stationary")
else:
    print("Fail to reject H0: Data is non-stationary")



# Stationary / Non-Stationary???




In [ ]:
# KPSS Test
def kpss_test(series, regression='c'):
    result_test = kpss(series, regression=regression)
    p_value = result_test[1]
    print(f"KPSS Statistic: {result_test[0]}")
    print(f"P-value: {result_test[1]}")
    print(f"Critical Values: {result_test[3]}")
    
    if p_value < 0.05:
        print("Reject H0: Data is non-stationary")
    else:
        print("Fail to Reject H0: Data is stationary")

    kpss_test(data['FEDFUNDS'])

In [ ]:
# ARIMAX Model Adjustments
data['observation_date'] = np.arange(len(data))
x = data[['observation_date']]

arimax_model = ARIMA(data['FEDFUNDS'], exog = x, order=(1, 1, 1))
arimax_fit = arimax_model.fit()

print(arimax_fit.summary())

# Check Residuals
residuals = arimax_fit.resid
kpss_test(residuals)


# Plotting the Results
plt.figure(figsize=(10, 6))
plt.plot(data.index, data['FEDFUNDS'], label='Actual')
plt.plot(data.index, arimax_fit.fittedvalues, label='Fitted', linestyle='--')
plt.legend()
plt.show()

# Plotting the Residuals
plt.figure(figsize=(10, 6))
plt.plot(residuals)
plt.title('Residuals of ARIMAX Model')
plt.show()

In [ ]:
# Forecasting
forecast_steps = 300
future_time = np.arange(len(data), len(data) + forecast_steps)
future_exog = pd.DataFrame(future_time, columns=['observation_date'])

forecast_values = arimax_fit.forecast(steps=forecast_steps, exog=future_exog)

print("Forecasted Values:", forecast_values)

# Plot Forecast
forecast_dates = pd.date_range(start=data.index[-1], periods=forecast_steps + 1, freq='M')[1:]

plt.figure(figsize=(12, 8))
plt.plot(data.index, data['FEDFUNDS'], label='Actual')
plt.plot(forecast_dates, forecast_values, label='Forecast', linestyle='--', color='red')
plt.legend()
plt.title('Actual vs Forecasted FEDFUNDS')
plt.show()